# BOS and recurrence: a paired comparison

How do attention similarity and the stages of inference change when a
beginning-of-sequence token is prepended? Compare **no BOS** and **+BOS** for the
same text, model, recurrence count, dataset indices, and seed. The input audit
checks that +BOS adds exactly one leading token, preserving native content and
trailing special tokens. No chat template is applied.

The explicit pairs below retain the six recurrent models at eight recurrences
and their three shared feedforward baselines. Both modes use these same pairs
and figures. `mode="quick"` defaults to one seeded example and saves figures only
in the run directory. `mode="full"` defaults to 256 examples and also updates the
paper figures. SLURM selects full mode; an explicit `limit` overrides the sample
count without changing the export policy.

In [ ]:
mode = "quick"  # "quick" saves locally; "full" also updates paper figures.
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
model_recurrences = [
    ("ouro_1_4b", 8),
    ("ouro_2_6b", 8),
    ("retrofitted_llama", 8),
    ("retrofitted_olmo", 8),
    ("retrofitted_tinyllama", 8),
    ("raven", 8),
    ("llama_1b", 1),  # Feedforward baselines: one pass each, reused below.
    ("olmo2", 1),
    ("tinyllama", 1),
]
limit = None  # Default: 1 (quick), 256 (full); 0 means the entire split.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import csv
import json
from dataclasses import asdict, replace
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch

from experiments.cache import ExperimentCache
from experiments.support import (
    BASELINE_FOR,
    BASELINES,
    MODEL_LABELS,
    add_prelude_coda_connectors,
    collect_metrics,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    save_figure,
    save_results,
)

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Papermill/YAML represents pairs as lists; normalize them for dictionary keys.
model_recurrences = [tuple(pair) for pair in model_recurrences]
if not model_recurrences or len(set(model_recurrences)) != len(model_recurrences):
    raise ValueError("Select at least one model-recurrence pair, without duplicates")

sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
configs = {
    (name, recurrences): configure_experiment(
        experiment="bos_comparison",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))
CONDITIONS = {"no_bos": "No BOS", "bos": "+BOS"}

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
METRICS = {
    "sink_rates": "Sink rate",
    "mixing_scores": "Mixing score",
    "colsum_concentrations": "Column concentration",
    "residual_entropy": "Residual entropy",
    "mean_sink_score": "Mean attention to first token",
    "residual_norms": "First-token residual norm",
}
PANELS = list(METRICS)[:4]
ATTENTION = "pairwise_attn_pattern_diff_norm"
NORMALIZED_ATTENTION = "pairwise_attn_pattern_diff_norm_per_sqrt_token"


def summarize_bos_comparison(run):
    """Keep per-example matrices, stage curves and exact inputs for pairing."""
    curves = {}
    for field in METRICS:
        values = run.get_metric(
            "sink_scores" if field == "mean_sink_score" else field
        ).float()
        if values.ndim > 1:
            values = values.mean(-1)
        if not bool(values.isfinite().all()):
            raise ValueError(f"Non-finite {field}")
        curves[field] = values.cpu().tolist()
    pairwise_attn_pattern_diff_norm = run.pairwise_attn_pattern_diff_norm.float().mean(
        -1
    )
    if not bool(pairwise_attn_pattern_diff_norm.isfinite().all()):
        raise ValueError("Non-finite pairwise_attn_pattern_diff_norm")
    return {
        "input_ids": run.input_ids.flatten().tolist(),
        "layers": [b.layer_index for b in run.per_virtual_layer_metrics],
        "virtual_layers": [
            b.virtual_layer_index for b in run.per_virtual_layer_metrics
        ],
        "recurrent_layers": run.recurrent_layer_indices,
        "pairwise_attn_pattern_diff_norm": pairwise_attn_pattern_diff_norm.tolist(),
        "pairwise_attn_pattern_diff_norm_per_sqrt_token": (
            pairwise_attn_pattern_diff_norm / run.input_ids.numel() ** 0.5
        ).tolist(),
        "curves": curves,
    }


def aggregate_bos(examples):
    """Mean matrices and stage means / sample SD; each example has equal weight."""
    topology = {
        key: examples[0][key]
        for key in ["layers", "virtual_layers", "recurrent_layers"]
    }
    if any(any(ex[key] != value for key, value in topology.items()) for ex in examples):
        raise ValueError("Different block topology across examples")
    curves = {}
    for field in METRICS:
        values = np.asarray([ex["curves"][field] for ex in examples])
        curves[field] = {
            "mean": values.mean(0),
            "std": values.std(0, ddof=int(len(examples) > 1)),
        }
    return {
        **topology,
        "n_examples": len(examples),
        "curves": curves,
        **{
            field: np.mean([ex[field] for ex in examples], axis=0)
            for field in [
                "pairwise_attn_pattern_diff_norm",
                "pairwise_attn_pattern_diff_norm_per_sqrt_token",
            ]
        },
    }


def recurrence_groups(data):
    layers = np.asarray(data["layers"])
    core = data["recurrent_layers"]
    if not core:
        return {"Feedforward": np.arange(len(layers))}
    groups = {}
    if np.any(layers < min(core)):
        groups["Prelude"] = np.flatnonzero(layers < min(core))
    virtual_layers_by_layer = [np.flatnonzero(layers == layer) for layer in core]
    if len({len(indices) for indices in virtual_layers_by_layer}) != 1:
        raise ValueError("Unequal recurrence counts")
    for recurrence in range(len(virtual_layers_by_layer[0])):
        groups[f"R{recurrence + 1}"] = np.asarray(
            [indices[recurrence] for indices in virtual_layers_by_layer]
        )
    if np.any(layers > max(core)):
        groups["Coda"] = np.flatnonzero(layers > max(core))
    return groups

In [ ]:
def check_input_pairs(pair, summaries, run_info):
    """Require the same examples and topology, with exactly one added BOS token."""
    name, recurrences = pair
    no_bos_info = run_info["no_bos"][pair]
    bos_info = run_info["bos"][pair]
    if no_bos_info["indices"] != bos_info["indices"]:
        raise ValueError(f"Different dataset indices between conditions: {pair}")
    bos_id = bos_info["tokenization"][name]["bos_token_id"]
    checks = []
    for index, no_bos, bos in zip(
        no_bos_info["indices"],
        summaries["no_bos"][pair],
        summaries["bos"][pair],
        strict=True,
    ):
        if bos_id is None or bos["input_ids"] != [bos_id, *no_bos["input_ids"]]:
            raise ValueError(
                f"BOS is not the sole input difference: {pair}, example {index}"
            )
        for key in ("layers", "virtual_layers", "recurrent_layers"):
            if no_bos[key] != bos[key]:
                raise ValueError(f"Different virtual layer topology: {pair}, {key}")
        checks.append(
            {
                "model": name,
                "recurrences": recurrences,
                "dataset_index": index,
                "sole_difference_is_bos": True,
            }
        )
    return checks

## Generate and reduce

Collect each pair once per BOS condition on the same seeded examples. Shared
baselines run once per condition and are reused in the recurrence profiles.
Pairwise attention and residual entropy are both needed by the figures below;
convergence metrics are disabled. After each evaluation, retain the reduced
matrices, curves, exact inputs, and run metadata, then release the model and
collected metrics.

`overview.pt` stores these per-example summaries and tokenizer provenance.
`paired-input-checks.json` records the input audit; `stage-curves.csv` stores the
aggregate means and sample standard deviations. All later figure cells reuse
the summaries without repeating inference.

In [ ]:
settings = asdict(config)
for key in ("models", "num_steps", "bos_policy", "include_baselines"):
    del settings[key]
settings.update(
    model_recurrences=model_recurrences,
    conditions=list(CONDITIONS),
    output_dir=str(config.output_dir),
)
settings["versions"] = {
    name: version(name) for name in ("torch", "transformer-lens", "transformers")
}
settings["cache"] = result_cache.settings
run_dir = save_results({}, settings, config.output_dir)

summaries = {condition: {} for condition in CONDITIONS}
run_info = {condition: {} for condition in CONDITIONS}
checks = []
for pair, pair_config in configs.items():
    name, recurrences = pair
    for condition in CONDITIONS:
        results, info = collect_metrics(
            replace(pair_config, bos_policy=condition),
            summarize_bos_comparison,
            cache=result_cache,
            analysis_version=analysis_version,
        )
        summaries[condition][pair] = results[name]
        run_info[condition][pair] = info
        del results
    checks.extend(check_input_pairs(pair, summaries, run_info))

aggregated = {
    condition: {pair: aggregate_bos(examples) for pair, examples in data.items()}
    for condition, data in summaries.items()
}
torch.save({"summaries": summaries, "run": run_info}, run_dir / "overview.pt")
(run_dir / "paired-input-checks.json").write_text(json.dumps(checks, indent=2))
print(f"Verified {len(checks)} input pairs; saved summaries to {run_dir}")
result_cache.write_manifest(run_dir)

In [ ]:
with (run_dir / "stage-curves.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(
        [
            "condition",
            "model",
            "recurrences",
            "metric",
            "virtual_layer",
            "layer",
            "n_examples",
            "mean",
            "std",
        ]
    )
    for condition, data in aggregated.items():
        for (name, recurrences), values in data.items():
            for field, curve in values["curves"].items():
                for virtual_layer, layer, mean, std in zip(
                    values["virtual_layers"],
                    values["layers"],
                    curve["mean"],
                    curve["std"],
                    strict=True,
                ):
                    writer.writerow(
                        [
                            condition,
                            name,
                            recurrences,
                            field,
                            virtual_layer,
                            layer,
                            values["n_examples"],
                            mean,
                            std,
                        ]
                    )

## What the measurements mean

| Measurement | Definition and interpretation |
| --- | --- |
| Attention distance | Mean over heads of the Frobenius distance between two virtual layers' attention matrices, computed within each condition. Smaller means more similar. |
| Length-normalized distance | Each example's attention distance divided by $\sqrt{T}$, with $T$ its input length, before averaging examples. |
| Sink rate | Fraction of heads whose mean attention to position zero is at least 0.3. |
| Mixing score | Attention-row entropy divided by its causal maximum, averaged over query rows excluding the first, then over heads. |
| Column concentration | One minus the normalized entropy of attention received across key positions, averaged over heads. |
| Residual entropy | $-\sum_i p_i\log p_i$, where $p_i$ is the normalized squared singular value of the token-by-feature residual matrix. Uses the complete spectrum. |
| Additional exported curves | Mean attention to position zero and the L2 norm of its residual stream. |

Average heads **within each example first**, then give every example equal
weight. Inputs include the likelihood task's context and continuation. Profile
shading is one sample standard deviation across examples, or zero for one
example; it is not a confidence interval. Residual entropy describes the residual
matrix, not the vocabulary distribution.

A virtual layer is one layer application in execution order. Recorded layer
identities determine prelude, recurrent passes, and coda. **R1 is the first
recurrent pass**. Layer depth divides each physical layer index by the maximum
index, placing the first and last layers at 0% and 100%.

Position zero changes token identity between conditions. BOS also adds a query
row and key column to attention: length normalization checks overall scaling,
not a content-only or causal-mechanism control. For checkpoints with stochastic
recurrent initialization, each sequence draws a fresh state. The seed is reset
at the start of each pair/condition, not each example; changing sequence shapes
can change the noise on corresponding content tokens and later random draws.
Thus example variation includes both text and initialization. Repeating seeds
helps assess small effects.

In [ ]:
def pair_title(pair):
    name, recurrences = pair
    suffix = "1 pass" if name in BASELINES else f"{recurrences} recurrences"
    return f"{MODEL_LABELS[name]} ({suffix})"


def plot_attention(pairs, field, filename):
    columns = min(3, len(pairs))
    rows = 2 * ((len(pairs) + columns - 1) // columns)
    fig, axes = plt.subplots(
        rows,
        columns,
        squeeze=False,
        figsize=get_figsize(len(pairs) > 1, (rows, columns), height_per_subplot=1.5),
        layout="constrained",
    )
    for ax in axes.flat:
        ax.set_visible(False)
    for index, pair in enumerate(pairs):
        row, col = 2 * (index // columns), index % columns
        vmax = max(float(aggregated[c][pair][field].max()) for c in CONDITIONS)
        title = pair_title(pair).replace(" (", "\n(")
        for offset, (condition, label) in enumerate(CONDITIONS.items()):
            ax = axes[row + offset, col]
            ax.set_visible(True)
            data = aggregated[condition][pair]
            matrix = data[field]
            artist = ax.imshow(
                matrix, origin="lower", cmap="viridis", vmin=0, vmax=vmax or 1
            )
            for indices in list(recurrence_groups(data).values())[1:]:
                ax.axhline(indices[0] - 0.5, color="white", ls="--", lw=0.4)
                ax.axvline(indices[0] - 0.5, color="white", ls="--", lw=0.4)
            ax.set(
                title=f"{title} · {label}" if offset == 0 else label,
                xlabel="Virtual layer",
                ylabel="Virtual layer",
            )
            ax.grid(False)
            name, recurrences = pair
            np.save(
                run_dir / f"{name}-{recurrences}-steps-{condition}-{field}.npy", matrix
            )
        fig.colorbar(
            artist,
            ax=list(axes[row : row + 2, col]),
            shrink=0.7,
            label="Attn. difference norm"
            if field == ATTENTION
            else "Attn. difference norm / √tokens",
        )
    save_figure(fig, run_dir, filename, bbox_inches=None)

In [ ]:
def draw_profiles(ax, data, field, baseline=None):
    groups = recurrence_groups(data)
    count = sum(key.startswith("R") for key in groups)
    positions = np.asarray(data["layers"], dtype=float)
    positions = positions / max(positions.max(), 1) * 100
    lines = {}
    for key, indices in groups.items():
        color = (
            {"Prelude": "darkorange", "Coda": "black", "Feedforward": "gray"}.get(key)
            if not key.startswith("R")
            else plt.colormaps["viridis"]((int(key[1:]) - 1) / max(count - 1, 1))
        )
        values = data["curves"][field]
        mean, std = values["mean"][indices], values["std"][indices]
        (lines[key],) = ax.plot(
            positions[indices],
            mean,
            color=color,
            label=key,
            linewidth=1.5 if key == "R1" else 1,
        )
        ax.fill_between(
            positions[indices], mean - std, mean + std, color=color, alpha=0.08
        )
    add_prelude_coda_connectors(ax, lines)
    if baseline is not None:
        name, reference = baseline
        x = np.asarray(reference["layers"], dtype=float)
        ax.plot(
            x / max(x.max(), 1) * 100,
            reference["curves"][field]["mean"],
            color="red",
            linestyle="--",
            linewidth=1,
            label=MODEL_LABELS[name],
        )
    ax.set(xlabel="Layer depth (%)", ylabel=METRICS[field])
    ax.grid(alpha=0.15)
    if field == "sink_rates":
        ax.set_ylim(-0.04, 1.04)


def baseline_for(pair, condition):
    baseline = (BASELINE_FOR.get(pair[0]), 1)
    return (
        (baseline[0], aggregated[condition][baseline])
        if baseline in aggregated[condition]
        else None
    )


def plot_profiles(pair):
    fig, axes = plt.subplots(
        2,
        len(PANELS),
        squeeze=False,
        sharex=True,
        sharey="col",
        figsize=get_figsize(True, (2, len(PANELS)), height_per_subplot=1.7),
        layout="constrained",
    )
    for row, (condition, label) in enumerate(CONDITIONS.items()):
        for col, field in enumerate(PANELS):
            draw_profiles(
                axes[row, col],
                aggregated[condition][pair],
                field,
                baseline_for(pair, condition),
            )
            axes[row, col].set_title(label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncols=min(len(labels), 6))
    fig.suptitle(pair_title(pair))
    name, recurrences = pair
    save_figure(
        fig,
        run_dir,
        f"{name}-{recurrences}-steps-recurrence-profiles",
        bbox_inches=None,
    )


def plot_profile_summary(pairs, field):
    columns = min(3, len(pairs))
    rows = 2 * ((len(pairs) + columns - 1) // columns)
    fig, axes = plt.subplots(
        rows,
        columns,
        squeeze=False,
        sharex=True,
        sharey=True,
        figsize=get_figsize(True, (rows, columns), height_per_subplot=1.5),
        layout="constrained",
    )
    for ax in axes.flat:
        ax.set_visible(False)
    legend_entries = {}
    for index, pair in enumerate(pairs):
        row, col = 2 * (index // columns), index % columns
        for offset, (condition, label) in enumerate(CONDITIONS.items()):
            ax = axes[row + offset, col]
            ax.set_visible(True)
            draw_profiles(
                ax, aggregated[condition][pair], field, baseline_for(pair, condition)
            )
            ax.set_title(f"{pair_title(pair)}\n{label}")
            handles, labels = ax.get_legend_handles_labels()
            legend_entries.update(zip(labels, handles, strict=True))
    fig.legend(
        list(legend_entries.values()),
        list(legend_entries),
        loc="outside lower center",
        ncols=min(len(legend_entries), 6),
    )
    save_figure(fig, run_dir, f"{field}-recurrence-summary", bbox_inches=None)

## Attention similarity

Each model's no-BOS and +BOS panels share a color scale. Dashed boundaries mark
prelude, recurrent passes, and coda. The overview includes all listed pairs;
individual raw and length-normalized comparisons follow at single-column width.

In [ ]:
plot_attention(model_recurrences, ATTENTION, "attention-similarity-summary")

In [ ]:
for pair in model_recurrences:
    name, recurrences = pair
    plot_attention(
        [pair], ATTENTION, f"{name}-{recurrences}-steps-attention-similarity"
    )

In [ ]:
for pair in model_recurrences:
    name, recurrences = pair
    plot_attention(
        [pair],
        NORMALIZED_ATTENTION,
        f"{name}-{recurrences}-steps-attention-length-normalized",
    )

## Stages of inference: recurrence profiles

Both conditions share axes and recurrence colors. Prelude and coda are labeled
separately; a red dashed line shows the matching feedforward baseline under the
same BOS condition, when its pair is listed. Individual figures retain all four
stage measurements, followed by one cross-model summary per measurement.

In [ ]:
for pair in model_recurrences:
    plot_profiles(pair)

In [ ]:
recurrent_pairs = [pair for pair in model_recurrences if pair[0] not in BASELINES]
if recurrent_pairs:
    plot_profile_summary(recurrent_pairs, "sink_rates")

In [ ]:
if recurrent_pairs:
    plot_profile_summary(recurrent_pairs, "mixing_scores")

In [ ]:
if recurrent_pairs:
    plot_profile_summary(recurrent_pairs, "colsum_concentrations")

In [ ]:
if recurrent_pairs:
    plot_profile_summary(recurrent_pairs, "residual_entropy")

## Interpretation

No-BOS follows Ouro's native tokenizer convention; for checkpoints that normally
prepend BOS it is an intervention. Conversely, +BOS is an intervention for
checkpoints that normally omit it. Each run's tokenizer probes record the
observed native policy; tokenizer defaults alone do not establish pretraining
formats.

Compare R1 with later recurrences within each condition before interpreting
stages of inference. These figures measure internal activity, not accuracy or
the usefulness of BOS. Consider the added attention dimension and stochastic
initial states alongside changes in model computation. Exact inputs and
per-example summaries remain available for further controls.

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")